# LTX-2.3 22B Distilled 1.1 Q4 — Video Generator

Self-contained Kaggle notebook for Text-to-Video and Image-to-Video generation with the LTX-2.3 22B distilled 1.1 Q4_K_M GGUF model via Wan2GP and Gradio.

GitHub: https://github.com/kcblak/LTX-2.3-22B-bulk-1.1-Q4-Video-Generator-by-blak

**Important:** After running Cell 2 (PyTorch install), restart the kernel (Kernel → Restart Kernel) and then Run All from the top. Do not skip the restart — PyTorch installed via pip into an already-imported `torch` session will not take effect.

## Step 1: Environment Setup

In [ ]:
import os
import psutil
import subprocess

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("RAM:")
print(f"  Total: {psutil.virtual_memory().total / 1024**3:.2f} GB")
print(f"  Available: {psutil.virtual_memory().available / 1024**3:.2f} GB")

print("\nDisk:")
subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/tmp"], check=False)

## Step 2: Install PyTorch (pinned) and Clone Wan2GP

In [ ]:
import os
import subprocess

if subprocess.run("nvidia-smi", shell=True).returncode != 0:
    print("\n⚠️  No NVIDIA GPU detected. In Kaggle, go to Settings → Accelerator → GPU T4 x1, then restart the session.")

if not os.path.isdir("Wan2GP"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DeepBeepMeep/Wan2GP.git"], check=True)
else:
    print("Wan2GP already cloned.")

subprocess.run([
    "pip", "install", "-q",
    "torch==2.3.1", "torchvision==0.18.1", "torchaudio==2.3.1",
    "--index-url", "https://download.pytorch.org/whl/cu121"
], check=True)

print("✅ PyTorch 2.3.1 installed. Now: Kernel → Restart Kernel, then Run All.")

## Step 3: Install Remaining Dependencies (run after kernel restart)

In [ ]:
from pathlib import Path
import subprocess
import torch

Path("constraints.txt").write_text(
    "torch==2.3.1\ntorchvision==0.18.1\ntorchaudio==2.3.1\n",
    encoding="utf-8",
)

subprocess.run([
    "pip", "install", "--timeout", "120", "--retries", "5", "-q",
    "-r", "Wan2GP/requirements.txt", "-c", "constraints.txt"
], check=True)

subprocess.run([
    "pip", "install", "--timeout", "120", "--retries", "5", "-q",
    "mmgp", "gradio", "gguf", "soundfile", "pydrive2",
    "google-auth", "google-auth-oauthlib", "google-api-python-client",
    "-c", "constraints.txt"
], check=True)

assert torch.__version__.startswith("2.3.1"), (
    f"torch was upgraded to {torch.__version__} by a later pip install — "
    f"this breaks the sm_60/P100 compatibility patches. "
    f"Re-run cell 1 (PyTorch install) and do NOT install any package that pulls a newer torch."
)
print(f"✅ torch version locked at {torch.__version__}")

## Step 4: Download Models

In [ ]:
import os
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO = "Abiray/LTX-2.3-22B-DISTILLED-1.1-GGUF"
COMPANION_REPO = "DeepBeepMeep/LTX-2"
MODEL_DIR = Path("Wan2GP/models")
TMP_DIR = Path("/kaggle/tmp/models")
CACHE_DIR = Path("/kaggle/tmp/hf_cache")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN") or None
download_kwargs = {"token": hf_token} if hf_token else {}

def _download(repo_id, filename, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f"✓ Already exists: {dest}")
        return
    print(f"Downloading {repo_id}/{filename} -> {dest}")
    try:
        downloaded = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=str(dest.parent),
            local_dir_use_symlinks=False,
            cache_dir=str(CACHE_DIR),
            **download_kwargs,
        )
        downloaded = Path(downloaded)
        if downloaded.resolve() != dest.resolve():
            if dest.exists():
                dest.unlink()
            shutil.move(str(downloaded), str(dest))
    except Exception as e:
        print(f"❌ Failed to download {repo_id}/{filename}: {e}")
        print("If rate-limited or private access is required, set HF_TOKEN as a Kaggle secret and re-run.")
        raise

def _symlink_if_needed(target, link_name):
    target = Path(target)
    link_name = Path(link_name)
    if link_name.exists():
        print(f"✓ Already linked: {link_name}")
        return
    try:
        link_name.symlink_to(target)
        print(f"✓ Linked {link_name} -> {target}")
    except OSError as e:
        print(f"⚠️ Could not create symlink for {link_name}: {e}")
        shutil.copy2(target, link_name)
        print(f"✓ Copied fallback: {link_name}")

_transformer_tmp = TMP_DIR / "ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf"
_transformer_link = MODEL_DIR / "ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf"
_download(REPO, "LTX-2.3-22B-distilled-1.1-Q4_K_M.gguf", _transformer_tmp)
_symlink_if_needed(_transformer_tmp, _transformer_link)

companion_big = [
    "ltx-2.3-22b-distilled-lora-384.safetensors",
    "ltx-2.3-22b_embeddings_connector.safetensors",
    "ltx-2.3-22b_text_embedding_projection.safetensors",
    "ltx-2.3-22b_vae.safetensors",
]
for filename in companion_big:
    tmp = TMP_DIR / filename
    link = MODEL_DIR / filename
    _download(COMPANION_REPO, filename, tmp)
    _symlink_if_needed(tmp, link)

companion_small = [
    "ltx-2.3-22b_audio_vae.safetensors",
    "ltx-2.3-22b_vocoder.safetensors",
    "ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
    "ltx-2.3-temporal-upscaler-x2-1.0.safetensors",
]
for filename in companion_small:
    _download(COMPANION_REPO, filename, MODEL_DIR / filename)

gemma_dir = MODEL_DIR / "gemma-3-12b-it-qat-q4_0-unquantized"
gemma_files = [
    "gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors",
    "added_tokens.json",
    "chat_template.json",
    "config_light.json",
    "generation_config.json",
    "preprocessor_config.json",
    "processor_config.json",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer.model",
    "tokenizer_config.json",
]
for filename in gemma_files:
    _download(COMPANION_REPO, f"gemma-3-12b-it-qat-q4_0-unquantized/{filename}", gemma_dir / filename)

for root, dirs, files in os.walk(MODEL_DIR):
    for d in list(dirs):
        if d == ".cache":
            shutil.rmtree(Path(root) / d, ignore_errors=True)
for root, dirs, files in os.walk(TMP_DIR):
    for d in list(dirs):
        if d == ".cache":
            shutil.rmtree(Path(root) / d, ignore_errors=True)

print("\nDisk after downloads:")
subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/tmp"], check=False)

## Step 5: Write the Inference + Gradio Script

In [ ]:
%%writefile run_ltx_t2v.py
import gc
import os
import sys
import json
import random
import tempfile
import glob
import traceback
import subprocess
import threading
import time
import csv
import io
import zipfile
import shutil
import numpy as np
import psutil
import soundfile as sf
from PIL import Image

# ---- bootstrap Wan2GP ----
WAN2GP_DIR = os.path.abspath("Wan2GP")
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.5"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

import torch
assert torch.__version__.startswith("2.3.1"), (
    f"run_ltx_t2v.py started with torch=={torch.__version__}, expected 2.3.1. "
    f"Re-run the PyTorch install cell and restart the kernel."
)

# Detect GPU architecture once — used throughout to gate sm_60-specific workarounds
_GPU_SM = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
_IS_SM60 = (_GPU_SM[0] == 6)   # P100 = sm_60; T4 = sm_75; A100 = sm_80; etc.
_IS_SM70_PLUS = (_GPU_SM[0] >= 7)

if _IS_SM60:
    # P100 (sm_60/Pascal) has NO native BF16 CUDA kernels and the current
    # PyTorch 2.4+ build dropped sm_60 support entirely.
    # Force FP16 to get as far as possible, and patch audio ops to run on CPU.
    os.environ["WGP_DTYPE"] = "fp16"
    print(f"  [GPU] sm_60 detected (P100) — FP16 mode + CPU audio patches enabled")
else:
    print(f"  [GPU] sm_{_GPU_SM[0]}{_GPU_SM[1]} detected — native CUDA mode, no patches needed")
import gradio as gr
from shared.utils.audio_video import save_video

GDRIVE_SA_JSON = os.environ.get("GDRIVE_SERVICE_ACCOUNT_JSON", "")
GDRIVE_SHARED_DRIVE_ID = os.environ.get("GDRIVE_SHARED_DRIVE_ID", "")
GDRIVE_AVAILABLE = False
_gdrive_service = None

def _init_gdrive():
    """Initialize a Google Drive v3 API client from a service-account JSON string."""
    global _gdrive_service, GDRIVE_AVAILABLE
    if not GDRIVE_SA_JSON:
        print("  [GDrive] No service account configured — backup disabled, bulk queue will keep local files.")
        return None
    try:
        from google.oauth2 import service_account
        from googleapiclient.discovery import build
        info = json.loads(GDRIVE_SA_JSON)
        creds = service_account.Credentials.from_service_account_info(
            info, scopes=["https://www.googleapis.com/auth/drive"]
        )
        service = build("drive", "v3", credentials=creds, cache_discovery=False)
        GDRIVE_AVAILABLE = True
        print("  [GDrive] ✅ Service account authenticated")
        return service
    except Exception as e:
        print(f"  [GDrive] ⚠️ Initialization failed: {e} — backup disabled, bulk queue will keep local files.")
        return None

_gdrive_service = _init_gdrive()

def upload_video_to_gdrive(video_path, folder_id=None):
    """Upload one video to Drive and return its Drive file ID, or None on failure."""
    if not GDRIVE_AVAILABLE or _gdrive_service is None or not video_path or not os.path.exists(video_path):
        return None
    try:
        from googleapiclient.http import MediaFileUpload
        filename = os.path.basename(video_path)
        metadata = {"name": filename}
        if folder_id:
            metadata["parents"] = [folder_id]
        media = MediaFileUpload(video_path, mimetype="video/mp4", resumable=True)
        kwargs = {"body": metadata, "media_body": media, "fields": "id"}
        if GDRIVE_SHARED_DRIVE_ID:
            kwargs["supportsAllDrives"] = True
        file = _gdrive_service.files().create(**kwargs).execute()
        return file.get("id")
    except Exception as e:
        print(f"  [GDrive] ❌ Upload failed for {video_path}: {e}")
        return None

def _get_or_create_gdrive_folder(folder_name="LTX_Videos_Backup"):
    """Find or create a Drive folder, scoped to the Shared Drive if configured."""
    if not GDRIVE_AVAILABLE:
        return None
    try:
        query = f"name='{folder_name}' and trashed=false and mimeType='application/vnd.google-apps.folder'"
        kwargs = {"q": query, "fields": "files(id, name)"}
        if GDRIVE_SHARED_DRIVE_ID:
            kwargs.update(
                corpora="drive",
                driveId=GDRIVE_SHARED_DRIVE_ID,
                includeItemsFromAllDrives=True,
                supportsAllDrives=True,
            )
        results = _gdrive_service.files().list(**kwargs).execute()
        files = results.get("files", [])
        if files:
            return files[0]["id"]
        metadata = {"name": folder_name, "mimeType": "application/vnd.google-apps.folder"}
        if GDRIVE_SHARED_DRIVE_ID:
            metadata["parents"] = [GDRIVE_SHARED_DRIVE_ID]
        kwargs2 = {"body": metadata, "fields": "id"}
        if GDRIVE_SHARED_DRIVE_ID:
            kwargs2["supportsAllDrives"] = True
        folder = _gdrive_service.files().create(**kwargs2).execute()
        return folder.get("id")
    except Exception as e:
        print(f"  [GDrive] ⚠️ Folder lookup/create failed: {e}")
        return None

GDRIVE_FOLDER_ID = _get_or_create_gdrive_folder() if GDRIVE_AVAILABLE else None
# ==== GGUF EXTENSION HANDLER ====
# mmgp uses an extension handler system for non-safetensors formats.
# The full Wan2GP app registers this internally; for standalone scripts
# we must register it ourselves before any model loading happens.

def _register_gguf_handler():
    """Register the GGUF handler with mmgp's quant_router."""
    import shared.qtypes.gguf
    print("  [GGUF] ✅ Extension handler registered with mmgp (Wan2GP native)")

def _patch_ltx2_config_loading():
    """Patch _load_config_from_checkpoint to handle GGUF metadata errors."""
    import models.ltx2.ltx2 as ltx2_mod
    _original = ltx2_mod._load_config_from_checkpoint

    def _patched(path, fallback_config_path=None):
        from mmgp import quant_router
        if isinstance(path, (list, tuple)):
            path = path[0] if path else ""
        if not path:
            return {}
        try:
            _, metadata = quant_router.load_metadata_state_dict(path)
            if metadata:
                config_raw = metadata.get("config")
                if config_raw:
                    config = ltx2_mod._normalize_config(config_raw)
                    if config:
                        return config
        except Exception as e:
            print(f"  [GGUF Patch] Metadata read: {type(e).__name__}, using fallback config")
        if fallback_config_path and os.path.isfile(fallback_config_path):
            try:
                with open(fallback_config_path, "r", encoding="utf-8") as f:
                    config = ltx2_mod._normalize_config(json.load(f))
                    if config:
                        print(f"  [GGUF Patch] ✅ Config loaded from {os.path.basename(fallback_config_path)}")
                        return config
            except Exception:
                pass
        return {}

    ltx2_mod._load_config_from_checkpoint = _patched
    print("  [GGUF Patch] ✅ Config loading patched for GGUF")


# ==== GPU INFO ====
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Compute Capability: {torch.cuda.get_device_capability()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
sys.stdout.flush()

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# ==== REGISTER GGUF + LOAD MODEL ====
print("\nLoading LTX-2.3 22B Distilled 1.1 (GGUF Q4_K_M)...")
sys.stdout.flush()

from mmgp import offload
from shared.utils import files_locator as fl

fl.set_checkpoints_paths(["models", "ckpts", "."])

from models.ltx2.ltx2_handler import family_handler

# Register GGUF handler + patch config loading BEFORE load_model
_register_gguf_handler()
_patch_ltx2_config_loading()


class _AudioEncoderP100Wrapper:
    """
    Wraps the audio encoder to fix cudaErrorNoKernelImageForDevice on P100.

    Root cause (ltx2.py line 1171-1176):
        audio_params = next(self.audio_encoder.parameters(), None)
        audio_device = audio_params.device   # returns 'cpu' under mmgp Profile 4
        mel = mel.to(device=audio_device, ...)  # mel goes to CPU
        audio_latent = self.audio_encoder(mel)  # mmgp moves encoder to CUDA
                                                # but mel is STILL on CPU
        -> Conv2d: weights on CUDA, input on CPU -> cudaErrorNoKernelImageForDevice

    Fix: intercept __call__ and move mel to CUDA FP16 BEFORE passing to encoder.
    All attribute access (sample_rate, mel_bins, parameters, etc.) is proxied.
    """
    def __init__(self, encoder):
        object.__setattr__(self, '_enc', encoder)

    def __call__(self, mel):
        if torch.cuda.is_available():
            mel = mel.to(
                device=torch.device("cuda", torch.cuda.current_device()),
                dtype=torch.float16,  # P100 (sm_60) has no BF16 CUDA kernels
            )
        return object.__getattribute__(self, '_enc')(mel)

    def __getattr__(self, name):
        return getattr(object.__getattribute__(self, '_enc'), name)

    def __setattr__(self, name, value):
        if name == '_enc':
            object.__setattr__(self, name, value)
        else:
            setattr(object.__getattribute__(self, '_enc'), name, value)

    def __repr__(self):
        return f"_AudioEncoderP100Wrapper({object.__getattribute__(self, '_enc')!r})"

base_model_type = "ltx2_22B"
model_def = {"ltx2_pipeline": "distilled"}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

gemma_folder = "models/gemma-3-12b-it-qat-q4_0-unquantized"
gemma_files = sorted(glob.glob(os.path.join(gemma_folder, "*.safetensors")))
quanto_files = [f for f in gemma_files if "quanto" in f]
text_encoder_file = quanto_files[0] if quanto_files else (gemma_files[0] if gemma_files else None)
if not text_encoder_file:
    raise FileNotFoundError(f"No .safetensors in {gemma_folder}. Check download cell.")
print(f"  Text encoder: {os.path.basename(text_encoder_file)}")

transformer_path = os.path.join("models", "ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf")
if not os.path.isfile(transformer_path):
    raise FileNotFoundError(f"{transformer_path} missing. Check download cell.")
print(f"  Transformer : {os.path.basename(transformer_path)}")
sys.stdout.flush()

# P100 (Pascal sm_60) has NO native BF16 support.
# Model weights loaded in FP16; but runtime activations (mel spectrogram) that
# flow through the BF16 autocast of the transformer are patched via CausalConv2d above.
# VAE_dtype = float32 for the video VAE (safer, audio VAE is patched separately).
MODEL_DTYPE = torch.float16
VAE_DTYPE   = torch.float16  # FP16 for T4: 65 TFLOPS vs FP32 8 TFLOPS = ~8x faster VAE decode

ltx2_model, pipe = family_handler.load_model(
    model_filename=transformer_path,
    model_type="ltx2_22B_distilled",
    base_model_type=base_model_type,
    model_def=model_def,
    dtype=MODEL_DTYPE,
    VAE_dtype=VAE_DTYPE,
    text_encoder_filename=text_encoder_file,
)

# ==== Verify pipeline components ====
print("\n--- Pipeline Components ---")
for name, component in pipe.items():
    if component is not None:
        ctype = type(component).__name__
        if hasattr(component, 'parameters'):
            try:
                p = next(component.parameters())
                print(f"  {name}: {ctype} (dtype={p.dtype})")
            except StopIteration:
                print(f"  {name}: {ctype} (no params)")
        else:
            print(f"  {name}: {ctype}")
    else:
        print(f"  {name}: None")
sys.stdout.flush()

# ==== sm_60 (P100) Only: patch CausalConv2d BEFORE offload.profile() ====
# On sm_75+ (T4, A100, etc.) this is skipped — native CUDA handles everything.
# On sm_60 the entire PyTorch CUDA kernel set is unavailable (2.4+ dropped it),
# so F.pad + conv must run on CPU. This patch must be applied AFTER load_model
# (class importable) but BEFORE offload.profile (mmgp captures previous_method).
if _IS_SM60:
    try:
        import torch.nn.functional as _F
        from models.ltx2.ltx_core.model.audio_vae.causal_conv_2d import CausalConv2d as _CC2d

        def _cc2d_cpu_pad(self, x: torch.Tensor) -> torch.Tensor:
            if x.is_cuda:
                dev, dt = x.device, x.dtype
                x_cpu = x.detach().cpu().float()
                x_cpu = _F.pad(x_cpu, self.padding)
                w = self.conv.weight.detach().cpu().float()
                b = self.conv.bias.detach().cpu().float() if self.conv.bias is not None else None
                out = _F.conv2d(x_cpu, w, b,
                                self.conv.stride, self.conv.padding,
                                self.conv.dilation, self.conv.groups)
                return out.to(device=dev, dtype=dt)
            else:
                x = _F.pad(x, self.padding)
                return self.conv(x)

        _CC2d.forward = _cc2d_cpu_pad
        print("  [sm_60 Fix] ✅ CausalConv2d patched: pad+conv run on CPU")
    except Exception as _e:
        print(f"  [sm_60 Fix] ❌ Could not patch CausalConv2d: {_e}")
else:
    print("  [sm_60 Fix] ⏭️  Skipped (not sm_60)")

# ==== Apply mmgp Profile 4 ====
print("\nApplying mmgp Profile 4 with per-model budgets...")
sys.stdout.flush()

offload.profile(
    pipe,
    profile_no=4,
    quantizeTransformer=False,
    convertWeightsFloatTo=torch.float16,
    budgets={
        # Budget 6000: ~9 min total (sweet spot — steps ~10s, VAE decode ~4 min)
        "transformer":       6000,
        "text_encoder":      1500,
        "video_encoder":     2000,
        "video_decoder":     3000,
        "audio_encoder":     1000,
        "audio_decoder":     1000,
        "vocoder":           500,
        "spatial_upsampler": 1500,
        "vae":               1000,
        "*":                 1000,
    },
)
offload.shared_state["_attention"] = "sdpa"

print("\n✅ Setup complete! Distilled 1.1 Text/Image-to-Video pipeline active.")
sys.stdout.flush()

# ==== HELPER FUNCTIONS ====
OUTPUT_DIR = "/kaggle/working/outputs"
_output_counter = 0

def list_outputs():
    if not os.path.isdir(OUTPUT_DIR):
        return []
    videos = [f for f in os.listdir(OUTPUT_DIR) if f.lower().endswith(('.mp4', '.mkv', '.webm'))]
    videos.sort(key=lambda x: os.path.getmtime(os.path.join(OUTPUT_DIR, x)), reverse=True)
    return videos

def _get_next_output_path():
    global _output_counter
    _output_counter += 1
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    return os.path.join(OUTPUT_DIR, f"ltx_video_{_output_counter:04d}.mp4")

def get_resolution(base_res_str, aspect_ratio_str):
    base_resolutions = {"1080p": 1088, "720p": 704, "540p": 544, "480p": 480}
    ratios = {
        "16:9 Landscape": 16/9, "4:3 Standard": 4/3,
        "1:1 Square": 1.0, "3:4 Portrait": 3/4, "9:16 Portrait": 9/16,
    }
    base = base_resolutions.get(base_res_str, 704)
    ratio = ratios.get(aspect_ratio_str, 16/9)
    if ratio >= 1.0:
        height = base
        width = int(base * ratio)
    else:
        width = base
        height = int(base / ratio)
    return (width // 32) * 32, (height // 32) * 32

def get_vae_tile_size(height, width):
    vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024**2)
    effective_vram = vram_mb / 1.5
    if effective_vram >= 24000: vae_config = 1
    elif effective_vram >= 8000: vae_config = 2
    else: vae_config = 3
    if max(height, width) > 480: vae_config += 1
    if vae_config <= 1: return 0
    elif vae_config == 2: return 512
    elif vae_config == 3: return 256
    return 128

def snap_to_ltx_frames(duration_sec: float, fps: float = 24.0, max_frames: int = 721) -> int:
    """Convert audio duration (seconds) to nearest valid LTX frame count.
    LTX distilled requires frames = 8k+1 (1, 9, 17, 25, ... 721, ...).
    Caps at max_frames (default 721 = 30s @ 24fps).
    """
    raw = duration_sec * fps
    # round to nearest 8k+1
    k = max(0, round((raw - 1) / 8))
    frames = 8 * k + 1
    frames = max(49, min(frames, max_frames))   # floor at 2s (49f), cap at max
    return int(frames)

@torch.inference_mode()
def Video_Generation(prompt, input_image_start, input_image_end, seed, duration_dropdown,
                     resolution_dropdown, aspect_ratio_dropdown,
                     guide_scale=3.0, num_steps=8, progress=gr.Progress()):
    try:
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        progress(0, desc="Starting...")

        duration_map = {
            "2 Seconds (49 frames)":  49,
            "3 Seconds (73 frames)":  73,
            "5 Seconds (121 frames)": 121,
            "8 Seconds (193 frames)": 193,
            "10 Seconds (241 frames)": 241,
            "15 Seconds (361 frames)": 361,
        }
        frame_rate = 24.0
        num_frames = duration_map.get(duration_dropdown, 121)
        width, height = get_resolution(resolution_dropdown, aspect_ratio_dropdown)

        if seed is None or seed < 0:
            seed = random.randint(0, 2**32 - 1)
        seed = int(seed)

        image_start = None
        image_end   = None
        if input_image_start is not None:
            image_start = Image.open(input_image_start).convert("RGB")
        if input_image_end is not None:
            image_end = Image.open(input_image_end).convert("RGB")

        free_vram = torch.cuda.mem_get_info()[0] / 1024**3
        ram = psutil.virtual_memory()
        mode = "T2V" if image_start is None else ("I2V first+last" if image_end else "I2V start")
        print(f"\n{'='*60}")
        print(f"Generating [{mode}]: {width}x{height}, {num_frames} frames, seed={seed}")
        print(f"  VRAM free: {free_vram:.2f} GB | RAM free: {ram.available / 1024**3:.1f} GB")
        print(f"  Prompt: {prompt[:120]}{'...' if len(prompt) > 120 else ''}")
        print(f"  Guide scale: {guide_scale}")
        print(f"{'='*60}")
        sys.stdout.flush()

        # Hardcode VAE tile size to 256 (matches audio notebook, prevents OOM/slowdowns at higher res)
        vae_tile_size = 256
        print(f"  VAE tile size: {vae_tile_size} (fixed)")

        total_steps = [8]
        current_step = [0]
        current_pass = [1]

        def cb(step, latent, is_start, override_num_inference_steps=None, pass_no=None, **kwargs):
            if is_start:
                if override_num_inference_steps is not None:
                    total_steps[0] = override_num_inference_steps
                if pass_no is not None:
                    current_pass[0] = pass_no
                current_step[0] = 0
                return
            current_step[0] += 1
            stage_name = (
                "Stage 1 (half-res)" if current_pass[0] == 1
                else "Stage 2 (full-res refine)" if current_pass[0] == 2
                else "Diffusion"
            )
            free_v = torch.cuda.mem_get_info()[0] / 1024**3
            print(f"  [{stage_name}] step {current_step[0]}/{total_steps[0]} | VRAM free: {free_v:.2f} GB")
            sys.stdout.flush()
            frac = current_step[0] / max(total_steps[0], 1)
            if current_pass[0] == 2:
                frac = 0.73 + 0.22 * frac
            else:
                frac = frac * 0.73
            progress(min(frac, 0.95), desc=f"{stage_name}: {current_step[0]}/{total_steps[0]}")

        _stage_labels = {
            "VAE Encoding": ("🇯️  VAE Encoding input frames...",   0.05),
            "VAE Decoding": ("🎦 VAE Decoding latents → frames...", 0.88),
            "Upsampling":   ("🔭 Spatial upsampling latents...",  0.83),
        }
        import time as _time
        _t = [_time.time()]

        def set_progress_status(status: str):
            dt = _time.time() - _t[0]; _t[0] = _time.time()
            label, frac = _stage_labels.get(status, (f"⏳ {status}...", 0.85))
            print(f"  [{status}] {label}  (+{dt:.1f}s)")
            sys.stdout.flush()
            progress(frac, desc=label)

        gen_kwargs = dict(
            input_prompt=prompt,
            image_start=image_start,
            height=height,
            width=width,
            frame_num=num_frames,
            fps=frame_rate,
            seed=seed,
            callback=cb,
            VAE_tile_size=vae_tile_size,
            input_video_strength=1.0,
            denoising_strength=1.0,
            guide_scale=float(guide_scale),
            sampling_steps=int(num_steps),
            guide_phases=2,
            n_prompt="",
            enhance_prompt=False,
            video_prompt_type="",
            audio_prompt_type="",
            set_progress_status=set_progress_status,
        )
        if image_end is not None:
            gen_kwargs["image_end"] = image_end

        print("  Diffusion starting → Stage 1 → spatial upsample → Stage 2 → VAE decode...")
        sys.stdout.flush()
        _t[0] = _time.time()

        result = ltx2_model.generate(**gen_kwargs)

        progress(0.97, desc="✅ Generation done — saving video...")
        print("  Pipeline complete.")
        sys.stdout.flush()

        if isinstance(result, dict):
            video_tensor = result.get("x")
            audio_data   = result.get("audio")
            audio_sr     = result.get("audio_sampling_rate", 24000)
        elif isinstance(result, tuple):
            video_tensor = result[0]
            audio_data   = result[1] if len(result) > 1 else None
            audio_sr     = result[2] if len(result) > 2 else 24000
        else:
            video_tensor = result
            audio_data, audio_sr = None, 24000

        if video_tensor is None or not torch.is_tensor(video_tensor):
            return None, f"❌ No video tensor. Got: {type(video_tensor)}"

        print(f"  Video tensor: {video_tensor.shape}, dtype={video_tensor.dtype}")
        video_tensor = video_tensor.cpu()
        gc.collect(); torch.cuda.empty_cache()

        out_path = _get_next_output_path()
        video_for_save = video_tensor.unsqueeze(0).float() / 127.5 - 1.0
        save_video(tensor=video_for_save, save_file=out_path, fps=frame_rate, normalize=True, value_range=(-1, 1))
        print(f"  ✅ Video saved: {out_path}")

        # ==== Mux native model audio (if generated) ====
        if audio_data is not None:
            try:
                import soundfile as sf
                audio_tmp = tempfile.mktemp(suffix=".wav")
                if isinstance(audio_data, np.ndarray):
                    audio_np = audio_data
                    if audio_np.ndim == 2 and audio_np.shape[0] <= 2:
                        audio_np = audio_np.T
                    sf.write(audio_tmp, audio_np, int(audio_sr or 24000))
                elif torch.is_tensor(audio_data):
                    import torchaudio
                    cpu_audio = audio_data.cpu().float()
                    if cpu_audio.dim() == 1: cpu_audio = cpu_audio.unsqueeze(0)
                    if cpu_audio.dim() == 3: cpu_audio = cpu_audio.squeeze(0)
                    torchaudio.save(audio_tmp, cpu_audio, int(audio_sr or 24000))
                else:
                    raise ValueError(f"Unknown audio type: {type(audio_data)}")

                final_path = out_path.replace(".mp4", "_with_audio.mp4")
                subprocess.run([
                    "ffmpeg", "-y", "-i", out_path, "-i", audio_tmp,
                    "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                    "-shortest", final_path
                ], check=True, capture_output=True)
                if os.path.exists(final_path) and os.path.getsize(final_path) > 0:
                    out_path = final_path
                    print(f"  ✅ Native audio muxed into output")
                else:
                    print(f"  ⚠️ Audio mux produced empty file, using video-only")
            except Exception as e:
                print(f"  ⚠️ Audio mux failed: {e}")

        del video_tensor, video_for_save
        gc.collect(); torch.cuda.empty_cache()
        progress(1.0, desc="Done!")
        return out_path, f"✅ Done! Seed: {seed} | {width}x{height} | {num_frames} frames"

    except Exception as e:
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()
        return None, f"❌ Error: {str(e)}"

# ==== GRADIO UI (AIQUEST BRANDED) ====
CSS = """@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #6a11cb 0%, #2575fc 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 700; font-size: 15px; text-decoration: none; display: inline-block; color: white; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
button.primary { background: linear-gradient(135deg, #6a11cb 0%, #2575fc 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }
"""


# ═══════════════════════════════════════════════════════════════════════════════
#  BULK PROCESSING HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

_bulk_stop_flag = threading.Event()

def _parse_bulk_csv(csv_text: str):
    """Parse the bulk CSV per the schema in spec §5. Returns a list of dicts."""
    jobs = []
    reader = csv.DictReader(io.StringIO(csv_text.strip()))
    for row in reader:
        row = {k.strip().lower(): v.strip() if isinstance(v, str) else v for k, v in row.items() if k is not None}
        jobs.append({
            "prompt":       row.get("prompt", ""),
            "start_image":  row.get("start_image", row.get("start image", "")),
            "end_image":    row.get("end_image",   row.get("end image",   "")),
            "duration":     row.get("duration",    ""),
            "resolution":   row.get("resolution",  ""),
            "aspect_ratio": row.get("aspect_ratio", row.get("aspect ratio", "")),
            "seed":         int(row["seed"]) if row.get("seed", "").strip() not in ("", None) else -1,
            "guide_scale":  float(row["guide_scale"]) if row.get("guide_scale", row.get("guide scale", "")).strip() not in ("", None) else None,
            "steps":        int(row["steps"]) if row.get("steps", "").strip() not in ("", None) else None,
        })
    return jobs

def _extract_images_zip(zip_path, dest_dir):
    """Unzip the images archive, return {basename: full_path}."""
    os.makedirs(dest_dir, exist_ok=True)
    img_map = {}
    if zip_path and os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path) as zf:
            for name in zf.namelist():
                if name.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
                    zf.extract(name, dest_dir)
                    img_map[os.path.basename(name)] = os.path.join(dest_dir, name)
    return img_map

def run_bulk_queue_with_auto_delete(csv_file, zip_file,
                                     default_duration, default_resolution, default_aspect,
                                     default_guide, default_steps,
                                     progress=gr.Progress()):
    """The ONE bulk-queue runner. Upload to Drive first, then delete locally only on success."""
    global _bulk_stop_flag
    _bulk_stop_flag.clear()

    if csv_file is None:
        return "❌ Please upload a CSV file."

    try:
        with open(csv_file, "r", encoding="utf-8-sig") as f:
            jobs = _parse_bulk_csv(f.read())
    except Exception as e:
        return f"❌ CSV parse error: {e}"
    if not jobs:
        return "❌ No jobs found in CSV."

    img_dir = "/kaggle/working/bulk_images"
    img_map = _extract_images_zip(zip_file, img_dir) if zip_file else {}

    total, passed, failed = len(jobs), [], []
    log_lines = [f"🚀 Starting {total} job(s)…", ""]

    def _resolve(name):
        if not name:
            return None
        for candidate in (name, os.path.join("/kaggle/working", name), img_map.get(os.path.basename(name), "")):
            if candidate and os.path.exists(candidate):
                return candidate
        return None

    for idx, job in enumerate(jobs):
        if _bulk_stop_flag.is_set():
            log_lines.append("🛑 Stopped by user.")
            break

        progress(idx / total, desc=f"Job {idx+1}/{total}: {job['prompt'][:50]}…")
        log_lines.append(f"[{idx+1}/{total}] {job['prompt'][:80]}")

        try:
            out_path, status = Video_Generation(
                prompt=job["prompt"],
                input_image_start=_resolve(job["start_image"]),
                input_image_end=_resolve(job["end_image"]),
                seed=job["seed"],
                duration_dropdown=job["duration"] or default_duration,
                resolution_dropdown=job["resolution"] or default_resolution,
                aspect_ratio_dropdown=job["aspect_ratio"] or default_aspect,
                guide_scale=job["guide_scale"] if job["guide_scale"] is not None else default_guide,
                num_steps=job["steps"] if job["steps"] is not None else default_steps,
                progress=progress,
            )
        except Exception as e:
            failed.append(job["prompt"])
            log_lines.append(f"  ❌ Exception: {e}")
            gc.collect(); torch.cuda.empty_cache()
            continue

        if not (out_path and os.path.exists(out_path)):
            failed.append(job["prompt"])
            log_lines.append(f"  ❌ {status}")
            gc.collect(); torch.cuda.empty_cache()
            continue

        passed.append(out_path)
        log_lines.append(f"  ✅ Generated: {os.path.basename(out_path)}")

        if GDRIVE_AVAILABLE:
            file_id = upload_video_to_gdrive(out_path, GDRIVE_FOLDER_ID)
            if file_id:
                log_lines.append(f"  ✅ Backed up to Google Drive ({file_id})")
                try:
                    os.remove(out_path)
                    log_lines.append("  🗑️  Auto-deleted from local storage (backup confirmed)")
                except Exception as e:
                    log_lines.append(f"  ⚠️  Backed up but could not delete local copy: {e}")
            else:
                log_lines.append("  ⚠️  Drive upload failed — keeping local copy for safety")
        else:
            log_lines.append("  ℹ️  Google Drive not configured — keeping local copy")

        gc.collect(); torch.cuda.empty_cache()

    progress(1.0, desc="✅ Batch complete!")
    log_lines.append("")
    log_lines.append(f"{'═'*50}")
    log_lines.append(f"✅ Completed: {len(passed)}  ❌ Failed: {len(failed)}  Total: {total}")
    log_lines.append(f"{'═'*50}")

    if os.path.exists(img_dir):
        shutil.rmtree(img_dir, ignore_errors=True)

    return "\n".join(log_lines)

def stop_bulk():
    _bulk_stop_flag.set()
    return "🛑 Stop signal sent — current job will finish, then the queue halts."


def delete_selected_output(selected):
    if not selected:
        return list_outputs(), "❌ No video selected."
    path = os.path.join(OUTPUT_DIR, selected)
    if os.path.exists(path):
        try:
            os.remove(path)
            return list_outputs(), f"✅ Deleted {selected}"
        except Exception as e:
            return list_outputs(), f"❌ Delete failed: {e}"
    return list_outputs(), "❌ File not found."


def delete_all_outputs():
    if not os.path.isdir(OUTPUT_DIR):
        return [], "✅ No outputs to delete."
    for f in os.listdir(OUTPUT_DIR):
        if f.lower().endswith(('.mp4', '.mkv', '.webm')):
            try:
                os.remove(os.path.join(OUTPUT_DIR, f))
            except Exception:
                pass
    return list_outputs(), "✅ All output videos deleted."


def backup_video_to_gdrive(selected):
    if not selected:
        return "❌ No video selected."
    path = os.path.join(OUTPUT_DIR, selected)
    if not os.path.exists(path):
        return "❌ File not found."
    if not GDRIVE_AVAILABLE:
        return "⚠️ Google Drive is not configured — local file retained."
    file_id = upload_video_to_gdrive(path, GDRIVE_FOLDER_ID)
    if not file_id:
        return "❌ Drive backup failed — local file retained."
    return f"✅ Backed up {selected} to Google Drive ({file_id})."

def backup_all_videos_to_gdrive():
    videos = list_outputs()
    if not videos:
        return "✅ No output videos to back up."
    if not GDRIVE_AVAILABLE:
        return "⚠️ Google Drive is not configured — local files retained."
    backed_up = 0
    for selected in videos:
        path = os.path.join(OUTPUT_DIR, selected)
        file_id = upload_video_to_gdrive(path, GDRIVE_FOLDER_ID)
        if file_id:
            backed_up += 1
    return f"✅ Backed up {backed_up}/{len(videos)} output video(s) to Google Drive."

def delete_selected_with_gdrive_backup(selected):
    if not selected:
        return list_outputs(), "❌ No video selected."
    path = os.path.join(OUTPUT_DIR, selected)
    if not os.path.exists(path):
        return list_outputs(), "❌ File not found."
    if not GDRIVE_AVAILABLE:
        return list_outputs(), "⚠️ Google Drive is not configured — local file retained."
    file_id = upload_video_to_gdrive(path, GDRIVE_FOLDER_ID)
    if not file_id:
        return list_outputs(), "❌ Drive backup failed — local file retained."
    try:
        os.remove(path)
        return list_outputs(), f"✅ Deleted {selected} after Drive backup ({file_id})."
    except Exception as e:
        return list_outputs(), f"❌ Backed up but delete failed: {e}"

def delete_all_with_gdrive_backup():
    videos = list_outputs()
    if not videos:
        return [], "✅ No output videos to delete."
    if not GDRIVE_AVAILABLE:
        return list_outputs(), "⚠️ Google Drive is not configured — local files retained."
    deleted = 0
    for selected in videos:
        path = os.path.join(OUTPUT_DIR, selected)
        file_id = upload_video_to_gdrive(path, GDRIVE_FOLDER_ID)
        if file_id and os.path.exists(path):
            try:
                os.remove(path)
                deleted += 1
            except Exception:
                pass
    return list_outputs(), f"✅ Deleted {deleted}/{len(videos)} output video(s) after Drive backup."

with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="LTX-2.3 22B Video Generator | AIQUEST") as demo:
    gr.HTML('<div class="brand-header"><div class="brand-title">🎬 LTX-2.3 22B Distilled 1.1 Q4 — Video Generator</div><div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> | Kaggle T4 GPU</div><div class="social-buttons"><a href="https://youtube.com/@aiquestacademy" target="_blank" class="social-btn youtube-btn">▶️ Subscribe on YouTube</a><a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a></div></div>')

    with gr.Tabs():
        # ════════════════════════════════════════════════════════════════════════════════════
        # TAB 1: SINGLE VIDEO GENERATION
        # ════════════════════════════════════════════════════════════════════════════════════
        with gr.TabItem("🎬 Single Video", id="single"):
            gr.Markdown(
                "**Two-stage distilled 1.1 pipeline** (8 steps half-res → 2× upscale → 3 steps full-res) | Q4_K_M ~17.8GB\n\n"
                "💡 **Tip:** Leave image inputs empty for pure Text-to-Video. Add a start image for Image-to-Video."
            )

            with gr.Column():
                prompt = gr.Textbox(
                    label="🎨 Prompt", lines=3,
                    placeholder="A majestic eagle soaring over snowy mountain peaks at golden hour, cinematic, 4K..."
                )

                with gr.Accordion("🖼️ Image to Video (Optional)", open=False):
                    with gr.Row():
                        input_image_start = gr.Image(type="filepath", label="🎬 Start Frame (First Frame)")
                        input_image_end   = gr.Image(type="filepath", label="🎬 End Frame (Last Frame — Optional)")
                    gr.Markdown(
                        "*• **Start Frame only** → Image-to-Video (model generates from your image)*\n"
                        "*• **Both frames** → First+Last frame interpolation (model generates in-between)*\n"
                        "*• **Neither** → Pure Text-to-Video*"
                    )

                with gr.Row():
                    seed = gr.Number(label="🎲 Seed (-1 for Random)", value=-1, precision=0)
                    duration_dropdown = gr.Dropdown(
                        label="⏱️ Duration",
                        choices=[
                            "2 Seconds (49 frames)",
                            "3 Seconds (73 frames)",
                            "5 Seconds (121 frames)",
                            "8 Seconds (193 frames)",
                            "10 Seconds (241 frames)",
                            "15 Seconds (361 frames)",
                        ],
                        value="5 Seconds (121 frames)",
                    )

                with gr.Row():
                    resolution_dropdown = gr.Dropdown(
                        label="📐 Base Resolution",
                        choices=["1080p", "720p", "540p", "480p"],
                        value="720p",
                    )
                    aspect_ratio_dropdown = gr.Dropdown(
                        label="📏 Aspect Ratio",
                        choices=["16:9 Landscape", "4:3 Standard", "1:1 Square", "3:4 Portrait", "9:16 Portrait"],
                        value="16:9 Landscape",
                    )

                guide_scale = gr.Slider(
                    label="🎯 Prompt Strength (guide_scale)",
                    minimum=1.0, maximum=8.0, step=0.5, value=3.0,
                    info="3.0 = optimal for T2V | 4.0+ = strong prompt | 1–2 = free generation",
                )
                num_steps = gr.Slider(
                    label="⚡ Diffusion Steps",
                    minimum=2, maximum=8, step=1, value=8,
                    info="6 = faster | 8 = best quality (default)",
                )

                with gr.Row():
                    gen_btn   = gr.Button("🎬 Generate Video", variant="primary", size="lg", elem_id="gen-btn")
                    stop_btn  = gr.Button("🛑 Stop",            variant="secondary", size="lg", elem_id="stop-btn")
                    clear_btn = gr.Button("🗑️ Clear",           variant="secondary", size="lg", elem_id="clear-btn")
                video_out  = gr.Video(label="🎥 Output")
                status_out = gr.Textbox(label="ℹ️ Status", interactive=False)

                with gr.Accordion("🗂️ Output Manager", open=False):
                    gr.Markdown(f"**🔗 Google Drive Status:** {'✅ ENABLED - Videos will be backed up before deletion' if GDRIVE_AVAILABLE else '⚠️ DISABLED - local files are retained unless manually deleted'}")
                    
                    refresh_outputs_btn = gr.Button("🔄 Refresh Outputs")
                    outputs_dropdown = gr.Dropdown(
                        label="Generated Videos",
                        choices=[],
                        interactive=True
                    )
                    
                    with gr.Row():
                        backup_btn = gr.Button("📤 Backup to Google Drive", variant="primary")
                        delete_output_btn = gr.Button("🗑️ Delete with Backup", variant="stop")
                    
                    with gr.Row():
                        backup_all_btn = gr.Button("📤 Backup All to Drive")
                        delete_all_btn = gr.Button("⚠️ Delete All (after backup)", variant="stop")
                    
                    delete_status = gr.Textbox(label="Status", interactive=False)

                    refresh_outputs_btn.click(
                        fn=list_outputs,
                        outputs=[outputs_dropdown]
                    )
                    
                    backup_btn.click(
                        fn=backup_video_to_gdrive,
                        inputs=[outputs_dropdown],
                        outputs=[delete_status]
                    )
                    
                    backup_all_btn.click(
                        fn=backup_all_videos_to_gdrive,
                        outputs=[delete_status]
                    )

                    delete_output_btn.click(
                        fn=delete_selected_with_gdrive_backup,
                        inputs=[outputs_dropdown],
                        outputs=[outputs_dropdown, delete_status]
                    )

                    delete_all_btn.click(
                        fn=delete_all_with_gdrive_backup,
                        outputs=[outputs_dropdown, delete_status]
                    )

                gen_event = gen_btn.click(
                    fn=Video_Generation,
                    inputs=[prompt, input_image_start, input_image_end, seed, duration_dropdown,
                            resolution_dropdown, aspect_ratio_dropdown, guide_scale, num_steps],
                    outputs=[video_out, status_out],
                )
                stop_btn.click(fn=None, cancels=[gen_event])
                clear_btn.click(
                    fn=lambda: (None, None, None, "", -1),
                    outputs=[input_image_start, input_image_end, video_out, prompt, seed],
                )

        # ════════════════════════════════════════════════════════════════════════════════════
        # TAB 2: BULK QUEUE GENERATION
        # ════════════════════════════════════════════════════════════════════════════════════
        with gr.TabItem("📦 Bulk Queue", id="bulk"):
            gr.Markdown(
                "### 🚀 Bulk Video Generation\n\n"
                "**How to use:**\n"
                "1. **Create a CSV** with columns: `prompt, start_image, end_image, duration, resolution, aspect_ratio, seed, guide_scale, steps`\n"
                "2. **Zip your images** (PNG/JPG) with filenames matching the CSV\n"
                "3. **Upload CSV + ZIP**, set defaults for missing values\n"
                "4. **Click ▶️ Run Bulk Queue** → Videos auto-generate; Drive backup is attempted first, and local files are kept if backup is unavailable\n\n"
                "**Example CSV:**\n"
                "```\n"
                "prompt,start_image,end_image,duration,resolution,aspect_ratio,seed,guide_scale,steps\n"
                "Sunset over ocean,sunset_start.jpg,sunset_end.jpg,5 Seconds (121 frames),720p,16:9 Landscape,-1,3.0,8\n"
                "City timelapse,city.png,,3 Seconds (73 frames),540p,1:1 Square,42,2.5,6\n"
                "```"
            )

            with gr.Row():
                bulk_csv = gr.File(label="📄 CSV Manifest", file_types=[".csv"])
                bulk_zip = gr.File(label="🗜️ Images ZIP", file_types=[".zip"])

            gr.Markdown("**Default Settings** *(for rows with missing values):*")
            with gr.Row():
                bulk_dur = gr.Dropdown(
                    label="Duration",
                    choices=["2 Seconds (49 frames)","3 Seconds (73 frames)","5 Seconds (121 frames)","8 Seconds (193 frames)","10 Seconds (241 frames)","15 Seconds (361 frames)"],
                    value="5 Seconds (121 frames)"
                )
                bulk_res = gr.Dropdown(
                    label="Resolution",
                    choices=["1080p","720p","540p","480p"],
                    value="720p"
                )
                bulk_asp = gr.Dropdown(
                    label="Aspect Ratio",
                    choices=["16:9 Landscape","4:3 Standard","1:1 Square","3:4 Portrait","9:16 Portrait"],
                    value="16:9 Landscape"
                )

            with gr.Row():
                bulk_guide = gr.Slider(label="Guide Scale", minimum=1.0, maximum=8.0, step=0.5, value=3.0)
                bulk_step = gr.Slider(label="Steps", minimum=2, maximum=8, step=1, value=8)

            with gr.Row():
                bulk_btn = gr.Button("▶️ Run Bulk Queue", variant="primary", size="lg")
                bulk_stp = gr.Button("🛑 Stop", variant="stop", size="lg")

            bulk_log = gr.Textbox(label="📋 Process Log", lines=15, interactive=False, placeholder="Processing logs appear here...")

            bulk_event = bulk_btn.click(
                fn=run_bulk_queue_with_auto_delete,
                inputs=[bulk_csv, bulk_zip, bulk_dur, bulk_res, bulk_asp, bulk_guide, bulk_step],
                outputs=[bulk_log],
            )
            bulk_stp.click(fn=stop_bulk, outputs=[bulk_log])

    gr.HTML('<div class="footer"><p style="font-size: 16px; margin: 5px 0;">🎬 Created by <strong>AIQuest Academy</strong></p><p style="font-size: 14px; margin: 5px 0; color: #9ca3af;">Free &amp; Open Source | LTX-2.3 22B Distilled 1.1 Q4_K_M | Kaggle T4 GPU</p><p style="font-size: 13px; margin: 10px 0;"><a href="https://youtube.com/@aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">YouTube</a> | <a href="https://x.com/aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">X (Twitter)</a> | <a href="https://aiquest.site" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">aiquest.site</a></p></div>')

print("\nLaunching Gradio...")
sys.stdout.flush()

demo.queue()
demo.launch(share=True, inline=False, debug=True, show_error=True, max_threads=1, ssr_mode=False)


## Step 6: Google Drive Backup Setup (optional, for Bulk Queue auto-delete)

If you want bulk-generated videos backed up to Google Drive before local auto-deletion, add these Kaggle Secrets:

- `GDRIVE_SERVICE_ACCOUNT_JSON`: GCP service-account JSON key
- `GDRIVE_SHARED_DRIVE_ID`: Shared Drive ID, required for service-account uploads to a Shared Drive

If neither secret is present, the notebook runs normally without Drive backup. Bulk queue will keep generated videos locally instead of silently losing them.

In [ ]:
import os

GDRIVE_AVAILABLE = False
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    sa_json = secrets.get_secret("GDRIVE_SERVICE_ACCOUNT_JSON") or ""
    shared_drive_id = secrets.get_secret("GDRIVE_SHARED_DRIVE_ID") or ""
    os.environ["GDRIVE_SERVICE_ACCOUNT_JSON"] = sa_json
    os.environ["GDRIVE_SHARED_DRIVE_ID"] = shared_drive_id
    GDRIVE_AVAILABLE = bool(sa_json)
    print("✅ Google Drive secrets loaded from Kaggle Secrets." if GDRIVE_AVAILABLE else "ℹ️ Google Drive secrets not configured; local videos will be retained.")
except Exception as e:
    os.environ.setdefault("GDRIVE_SERVICE_ACCOUNT_JSON", "")
    os.environ.setdefault("GDRIVE_SHARED_DRIVE_ID", "")
    print(f"ℹ️ Google Drive secrets not configured or unavailable; local videos will be retained. ({e})")

## Step 7: Launch

In [ ]:
!cd /kaggle/working && python -u run_ltx_t2v.py 2>&1

## Footer

Created for LTX-2.3 22B Distilled 1.1 Q4 video generation on Kaggle.

GitHub: https://github.com/kcblak/LTX-2.3-22B-bulk-1.1-Q4-Video-Generator-by-blak